# SOCIOWEAVE — E1 Colab T4 infrastructure preflight

**Purpose:** verify an immutable repository checkout, an NVIDIA T4 runtime, local model loading/inference, and intake of the five operator-supplied frozen benchmark snapshots. This is an **infrastructure and data-intake preflight only**. It does not validate the scientific correctness of every strategy implementation, execute the E1 campaign, or generate scientific results. A passing notebook must not be reported as `E1_READY`.

Run only in a Colab runtime configured with a **T4 GPU**. Benchmark files must be obtained, versioned, split-selected, and license-reviewed separately; this notebook never downloads, selects, synthesizes, or approves benchmark prompts.

In [ ]:
import os, sys, json, subprocess, pathlib, platform, hashlib, importlib.metadata
from datetime import datetime, timezone
REPO_URL = 'https://github.com/msellamiTN/socioweave-studio.git'
CODE_SHA = '687272b493e039faf03e130b06236c0bf1578610'  # reviewed immutable commit
ROOT = pathlib.Path('/content/socioweave-e1')
REPO_DIR = ROOT / 'repo'
ARTIFACT_DIR = ROOT / 'artifacts'
DATA_DIR = ROOT / 'datasets'
for p in (ROOT, ARTIFACT_DIR, DATA_DIR): p.mkdir(parents=True, exist_ok=True)
print('Python:', sys.version)
print('Platform:', platform.platform())
print('UTC:', datetime.now(timezone.utc).isoformat())


In [ ]:
if not REPO_DIR.exists():
    subprocess.check_call(['git','clone','--no-checkout',REPO_URL,str(REPO_DIR)])
subprocess.check_call(['git','-C',str(REPO_DIR),'fetch','origin',CODE_SHA])
subprocess.check_call(['git','-C',str(REPO_DIR),'checkout','--detach',CODE_SHA])
actual_sha = subprocess.check_output(['git','-C',str(REPO_DIR),'rev-parse','HEAD'],text=True).strip()
assert actual_sha == CODE_SHA, (actual_sha, CODE_SHA)
print('Pinned repository SHA:', actual_sha)
subprocess.check_call([sys.executable,'-m','pip','install','-e',str(REPO_DIR / 'engine') + '[local]'])


In [ ]:
import torch
assert torch.cuda.is_available(), 'CUDA GPU is required; select a Colab GPU runtime.'
gpu_name = torch.cuda.get_device_name(0)
gpu_props = torch.cuda.get_device_properties(0)
gpu_total_gib = gpu_props.total_memory / (1024**3)
print('GPU:', gpu_name)
print('GPU memory GiB:', round(gpu_total_gib, 2))
assert 'T4' in gpu_name, f'Expected an NVIDIA T4 runtime, received: {gpu_name}'
assert gpu_total_gib >= 14.0, f'Unexpected GPU memory capacity: {gpu_total_gib:.2f} GiB'
print('CUDA:', torch.version.cuda, '| PyTorch:', torch.__version__)
print(subprocess.check_output(['nvidia-smi'],text=True))


## 1. Local model smoke test

The default is `Qwen/Qwen2.5-1.5B-Instruct` in 4-bit NF4 mode. This is an infrastructure smoke test only, not a paper-approved model assignment or E1 result. Record the exact model revision and license decision in the campaign preregistration before scientific use.

In [ ]:
from sociodec.providers.huggingface_local import HuggingFaceLocalProvider
from sociodec.providers.base import ChatMessage, GenerationConfig
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
provider = HuggingFaceLocalProvider(model_id=MODEL_ID, load_in_4bit=True, max_input_tokens=4096)
provider.load()
smoke = provider.generate(
    [ChatMessage('system','Return a short plain-text response.'), ChatMessage('user','Reply with exactly: SOCIOWEAVE_T4_OK')],
    GenerationConfig(temperature=0.0, max_new_tokens=24, seed=17),
)
assert smoke.text.strip(), 'Model returned an empty completion.'
print('Provider:', smoke.provider, '| Model:', smoke.model)
print('Smoke output:', repr(smoke.text))
print('Tokens in/out:', smoke.input_tokens, smoke.output_tokens)
print('Peak allocated GiB:', round(torch.cuda.max_memory_allocated()/(1024**3), 3))
print('Peak reserved GiB:', round(torch.cuda.max_memory_reserved()/(1024**3), 3))


## 2. Supply the five frozen dataset files

Place the operator-provided, license-reviewed JSONL files in `/content/socioweave-e1/datasets/` (or mount Drive and point `DATA_DIR` there). Each row must contain `example_id`, `prompt_text`, `split`, `metadata` and dataset-appropriate structured `reference_data` as required by Table 12. Use the exact source version, split and prompt-selection procedure approved for the study; do not assume `test` or select the first N rows. The intake enforces counts (100 each for BREAK, StrategyQA, TaskBench and TravelPlanner; 60 for smart-administration), preserves reference data and hashes provenance. The paper does not fully specify prompt IDs/sampling; document the operator's selection protocol explicitly.

### Optional: stage files from Google Drive

If the five reviewed JSONL files are stored in Drive, mount it and copy them into the notebook's expected `DATA_DIR`. The cell below copies only files that are present and does not overwrite existing local files. It does not download, create, repair, or approve datasets. If you use another location, set `DATA_DIR` explicitly before running the intake cell.

In [ ]:
from google.colab import drive
import shutil
drive.mount('/content/drive')

# Change this path if your files are stored elsewhere in Google Drive.
DRIVE_DATA_DIR = pathlib.Path('/content/drive/MyDrive/socioweave-e1/datasets')
EXPECTED_DATASET_NAMES = [
    'BREAK.jsonl', 'StrategyQA.jsonl', 'TaskBench.jsonl',
    'TravelPlanner.jsonl', 'smart-administration.jsonl',
]
staged, already_present, absent_in_drive = [], [], []
if not DRIVE_DATA_DIR.is_dir():
    print(f'OPTIONAL DRIVE STAGING: folder not found: {DRIVE_DATA_DIR}')
    print('No files were copied. This is not fatal; the dataset-intake cell will report BLOCKED for missing files.')
    print('Create the folder in Drive, upload the reviewed JSONL files, or edit DRIVE_DATA_DIR and rerun this cell.')
    absent_in_drive = EXPECTED_DATASET_NAMES.copy()
else:
    for filename in EXPECTED_DATASET_NAMES:
        source = DRIVE_DATA_DIR / filename
        destination = DATA_DIR / filename
        if destination.is_file():
            already_present.append(filename)
        elif source.is_file():
            shutil.copy2(source, destination)
            staged.append(filename)
        else:
            absent_in_drive.append(filename)
    print('Drive source folder:', DRIVE_DATA_DIR)
    print('Copied from Drive:', staged or 'none')
    print('Already present locally (not overwritten):', already_present or 'none')
    print('Not found in Drive:', absent_in_drive or 'none')
print('Dataset directory:', DATA_DIR)
print('Files currently present:', sorted(p.name for p in DATA_DIR.glob('*.jsonl')))


In [ ]:
from sociodec.datasets import E1DatasetSource, freeze_e1_benchmark_snapshot

DATASET_FILES = {
    'BREAK': DATA_DIR/'BREAK.jsonl',
    'StrategyQA': DATA_DIR/'StrategyQA.jsonl',
    'TaskBench': DATA_DIR/'TaskBench.jsonl',
    'TravelPlanner': DATA_DIR/'TravelPlanner.jsonl',
    'smart-administration': DATA_DIR/'smart-administration.jsonl',
}

# Replace these declarations only after verifying the exact source snapshots.
SOURCE_URLS = {name:'REPLACE_WITH_CANONICAL_SOURCE_URL' for name in DATASET_FILES}
SOURCE_REVISIONS = {name:'REPLACE_WITH_IMMUTABLE_UPSTREAM_REVISION' for name in DATASET_FILES}
ELIGIBLE_POPULATION_COUNTS = {name:0 for name in DATASET_FILES}
LICENSE_URLS = {name:'REPLACE_WITH_DATASET_LICENSE_OR_TERMS_URL' for name in DATASET_FILES}
LICENSE_IDENTIFIERS = {name:'REPLACE_WITH_EXACT_LICENSE_IDENTIFIER' for name in DATASET_FILES}
LICENSE_REVIEW_EVIDENCE = {name:'REPLACE_WITH_REVIEW_RECORD_OR_DECISION_REFERENCE' for name in DATASET_FILES}
LICENSE_REVIEWED_BY = {name:'REPLACE_WITH_REVIEWER_ID' for name in DATASET_FILES}
LICENSE_REVIEWED_AT = {name:'REPLACE_WITH_TIMEZONE_AWARE_ISO8601_TIMESTAMP' for name in DATASET_FILES}
SOURCE_VERSIONS = {name:'REPLACE_WITH_VERIFIED_SOURCE_VERSION' for name in DATASET_FILES}
SELECTED_SPLITS = {name:'REPLACE_WITH_VERIFIED_SPLIT' for name in DATASET_FILES}
SELECTION_PROTOCOLS = {name:'REPLACE_WITH_EXACT_SELECTION_PROTOCOL' for name in DATASET_FILES}
LICENSE_REVIEW_STATUS = {name:'REPLACE_WITH_APPROVED_STATUS' for name in DATASET_FILES}

# This cell is intentionally non-fatal for a missing operator dataset: infrastructure
# checks and the report can still complete, while dataset intake remains BLOCKED.
snapshot = None
snapshot_error = None
snapshot_status = 'BLOCKED'
missing = [name for name, path in DATASET_FILES.items() if not path.is_file()]
metadata_incomplete = (
    any('REPLACE_WITH_' in value for value in SOURCE_VERSIONS.values())
    or any('REPLACE_WITH_' in value for value in SELECTED_SPLITS.values())
    or any('REPLACE_WITH_' in value for value in SELECTION_PROTOCOLS.values())
    or any('REPLACE_WITH_' in value for value in SOURCE_URLS.values())
    or any('REPLACE_WITH_' in value for value in SOURCE_REVISIONS.values())
    or any(value < 1 for value in ELIGIBLE_POPULATION_COUNTS.values())
    or any('REPLACE_WITH_' in value for value in LICENSE_URLS.values())
    or any('REPLACE_WITH_' in value for value in LICENSE_IDENTIFIERS.values())
    or any('REPLACE_WITH_' in value for value in LICENSE_REVIEW_EVIDENCE.values())
    or any('REPLACE_WITH_' in value for value in LICENSE_REVIEWED_BY.values())
    or any('REPLACE_WITH_' in value for value in LICENSE_REVIEWED_AT.values())
)
license_unapproved = any(value != 'approved' for value in LICENSE_REVIEW_STATUS.values())

if missing:
    snapshot_error = 'Missing operator-supplied files: ' + ', '.join(missing)
    print('DATASET INTAKE: BLOCKED — required files are missing.')
    for name in missing:
        print(f'  - {name}: {DATASET_FILES[name]}')
    print('Stage the reviewed JSONL files, then rerun this cell.')
elif metadata_incomplete:
    snapshot_error = 'Source version, selected split, and selection protocol must be explicitly verified.'
    print('DATASET INTAKE: BLOCKED — provenance declarations are incomplete.')
elif license_unapproved:
    snapshot_error = 'Every dataset requires an explicit license_review_status of approved.'
    print('DATASET INTAKE: BLOCKED — license review is not approved for every dataset.')
else:
    try:
        sources = {
            name: E1DatasetSource(
                path=DATASET_FILES[name],
                source_version=SOURCE_VERSIONS[name],
                split=SELECTED_SPLITS[name],
                selection_protocol=SELECTION_PROTOCOLS[name],
                license_review_status=LICENSE_REVIEW_STATUS[name],
                source_url=SOURCE_URLS[name],
                source_revision=SOURCE_REVISIONS[name],
                eligible_population_count=ELIGIBLE_POPULATION_COUNTS[name],
                license_url=LICENSE_URLS[name],
                license_identifier=LICENSE_IDENTIFIERS[name],
                license_review_evidence=LICENSE_REVIEW_EVIDENCE[name],
                license_reviewed_by=LICENSE_REVIEWED_BY[name],
                license_reviewed_at=LICENSE_REVIEWED_AT[name],
            )
            for name in DATASET_FILES
        }
        snapshot = freeze_e1_benchmark_snapshot(sources)
        manifest_path = ARTIFACT_DIR/'e1_frozen_snapshot_manifest.json'
        manifest_path.write_text(snapshot.manifest_json(), encoding='utf-8')
        snapshot_status = 'PASS'
        print('DATASET INTAKE: PASS')
        print('Frozen prompts:', len(snapshot.examples))
        print('Snapshot SHA256:', snapshot.manifest['snapshot_sha256'])
        print('Manifest:', manifest_path)
    except Exception as exc:
        snapshot_error = f'{type(exc).__name__}: {exc}'
        snapshot_status = 'INVALID'
        print('DATASET INTAKE: INVALID — snapshot validation failed.')
        print(snapshot_error)

dataset_intake = {
    'status': snapshot_status,
    'expected_files': {name: str(path) for name, path in DATASET_FILES.items()},
    'missing_files': missing,
    'source_versions_declared': not any('REPLACE_WITH_' in value for value in SOURCE_VERSIONS.values()),
    'selected_splits_declared': not any('REPLACE_WITH_' in value for value in SELECTED_SPLITS.values()),
    'selection_protocols_declared': not any('REPLACE_WITH_' in value for value in SELECTION_PROTOCOLS.values()),
    'source_revisions_declared': not any('REPLACE_WITH_' in value for value in SOURCE_REVISIONS.values()),
    'license_evidence_declared': not any('REPLACE_WITH_' in value for value in LICENSE_REVIEW_EVIDENCE.values()),
    'all_licenses_approved': not license_unapproved,
    'error': snapshot_error,
    'snapshot_sha256': snapshot.manifest['snapshot_sha256'] if snapshot is not None else None,
    'prompt_count': len(snapshot.examples) if snapshot is not None else None,
}


In [ ]:
import transformers, accelerate

gate_status = {
    'repository_pin': 'PASS' if actual_sha == CODE_SHA else 'FAIL',
    't4_runtime': 'PASS' if ('T4' in gpu_name and gpu_total_gib >= 14.0) else 'FAIL',
    'local_model_smoke': 'PASS' if bool(smoke.text.strip()) else 'FAIL',
    'frozen_dataset_intake': snapshot_status,
    'scientific_e1_campaign': 'NOT_RUN',
    'e1_ready': False,
}

preflight = {
    'purpose':'infrastructure_and_dataset_intake_preflight_only_no_E1_trials',
    'gate_status': gate_status,
    'timestamp_utc':datetime.now(timezone.utc).isoformat(),
    'repository_url':REPO_URL,
    'code_sha':actual_sha,
    'gpu_name':gpu_name,
    'gpu_memory_gib':round(gpu_total_gib,3),
    'cuda_version':torch.version.cuda,
    'torch_version':torch.__version__,
    'transformers_version':transformers.__version__,
    'accelerate_version':accelerate.__version__,
    'model_id':MODEL_ID,
    'model_smoke_output':smoke.text,
    'model_smoke_input_tokens':smoke.input_tokens,
    'model_smoke_output_tokens':smoke.output_tokens,
    'dataset_intake': dataset_intake,
    'snapshot_sha256':snapshot.manifest['snapshot_sha256'] if snapshot is not None else None,
    'prompt_count':len(snapshot.examples) if snapshot is not None else None,
    'e1_ready': False,
}
preflight_path = ARTIFACT_DIR/'t4_preflight_report.json'
preflight_path.write_text(json.dumps(preflight,ensure_ascii=False,indent=2,sort_keys=True)+'\n',encoding='utf-8')
print(json.dumps(preflight,ensure_ascii=False,indent=2))
print('Saved:', preflight_path)
print('BOUNDARY: This report is not an E1 experiment result and never grants E1_READY.')


## Exit criteria and scientific boundary

Preflight passes only if: (1) checkout SHA matches exactly; (2) runtime is an NVIDIA T4 with expected memory; (3) local model loads and returns a non-empty response; (4) all five operator-supplied JSONL snapshots pass fail-closed intake and manifest hashing. Save both JSON artifacts with the run records.

**Not performed here:** E1 strategy/baseline campaign, fixed executor evaluation, V3/V4 assessment, human annotation, statistical analysis, or E1–E9 claims. Do not report this preflight as an experiment. The next gate is a deterministic fake-provider end-to-end runtime test, then an explicitly authorized and cost-capped provider smoke and bounded pilot. Neither is an E1 result.